In [31]:

%pip install python-dotenv gspread google-auth pandas requests


Note: you may need to restart the kernel to use updated packages.


In [32]:
# Cell 2: Imports & Environment Setup
import os
import json
from dotenv import load_dotenv
from google.oauth2.service_account import Credentials
import gspread
from datetime import datetime
import requests
import pandas as pd

# Load .env secrets
load_dotenv()

SERVICE_ACCOUNT_PATH = os.getenv('GOOGLE_SERVICE_ACCOUNT_PATH')
SPREADSHEET_ID = os.getenv('SPREADSHEET_ID')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

SCOPES = ['https://www.googleapis.com/auth/spreadsheets']

# Google Sheets Auth
creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_PATH, scopes=SCOPES)
client = gspread.authorize(creds)
sheet = client.open_by_key(SPREADSHEET_ID).sheet1

print("✅ Setup Complete!")


✅ Setup Complete!


In [33]:


name = input("Customer Name: ")

ITEM_PRICES = {
    "Milk": 40,
    "Egg": 8,
    "Apple": 30,
    "Bread": 20,
    "Banana": 7,
    "Rice": 1800,
}

order_items = []
print("\nAvailable items and prices:\n")
for item, price in ITEM_PRICES.items():
    print(f"  {item:10} : ₹{price}")
n = int(input("How many different items do you want?"))
for i in range(n):
    item = input(f"Item {i+1}: ")
    quantity = int(input(f"Quantity for {item}: "))
    unit_price = ITEM_PRICES.get(item, 0)
    total = unit_price * quantity
    order_items.append({"item": item, "quantity": quantity, "unit_price": unit_price, "total": total})

from datetime import datetime
order_date = datetime.now().strftime("%d-%m-%Y")
grand_total = sum(x['total'] for x in order_items)

def save_order_to_sheet(name, order_items, grand_total):
   
    for x in order_items:
        row = [name, x['item'], x['quantity'], x['total']]
        sheet.append_row(row)
   
    sheet.append_row([name, "Total", "", grand_total])
    print(f"Order saved to Google Sheet for: {name}")

save_order_to_sheet(name, order_items, grand_total)

item_lines = []
for x in order_items:
    item_lines.append(f"{x['item']} x {x['quantity']} units at ₹{x['unit_price']} each (₹{x['total']})")
items_text = "\n".join(item_lines)

order_desc = (
    "Please generate a polite, easy-to-read bill receipt for the customer in the following style:\n"
    "Bill Receipt\n"
    f"Date: {order_date}\n"
    f"Customer: {name}\n"
    "---------------------------\n"
    "Items Purchased:\n"
    + "\n".join([f"- {x['item']} Qty: {x['quantity']} x ₹{x['unit_price']} = ₹{x['total']}" for x in order_items]) +
    "\n---------------------------\n"
    f"Total Amount: ₹{grand_total}\n"
    "---------------------------\n"
    "Thank you for choosing our store, {name}. We appreciate your business. Have a great day!\n"
    "Please visit again!"
)

import requests

MODEL = "groq/compound"  

url = "https://api.groq.com/openai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}
payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are an English bill assistant. Always reply with a receipt in professional, friendly tone. Close with a thank-you and friendly wishes."},
        {"role": "user", "content": order_desc}
    ],
    "max_tokens": 250,
    "temperature": 0.6
}

response = requests.post(url, headers=headers, json=payload)
ai_reply = response.json()["choices"][0]["message"]["content"]
print("\nGroq AI Response:\n", ai_reply)





Available items and prices:

  Milk       : ₹40
  Egg        : ₹8
  Apple      : ₹30
  Bread      : ₹20
  Banana     : ₹7
  Rice       : ₹1800
Order saved to Google Sheet for: AYYAPPAN M

Groq AI Response:
 **Bill Receipt**  
Date: 06-01-2026  
Customer: AYYAPPAN M  

---------------------------  
**Items Purchased:**  
- Rice Qty: 7 × ₹1800 = ₹12600  
- Apple Qty: 8 × ₹30 = ₹240  
- Egg Qty: 15 × ₹8 = ₹120  

---------------------------  
**Total Amount:** ₹12960  

---------------------------  

Thank you for choosing our store, AYYAPPAN M. We appreciate your business. Have a great day!  
Please visit again!  

---  

Thank you and best wishes!


In [35]:

def create_pdf(ai_bill_text, name, order_date, grand_total):
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=letter)
    story = []
    styles = getSampleStyleSheet()
    
    
    story.append(Paragraph("*Bill Receipt*", styles['Title']))
    story.append(Spacer(1, 0.15*inch))
    story.append(Paragraph(f"Date: {order_date}", styles['Normal']))
    story.append(Paragraph(f"Customer: {name}", styles['Normal']))
    story.append(Spacer(1, 0.2*inch))
    
    
    story.append(Paragraph(ai_bill_text.replace('*', ''), styles['Normal']))
    
    doc.build(story)
    buffer.seek(0)
    return buffer


pdf_buffer = create_pdf(ai_reply, name, order_date, grand_total)
pdf_base64 = base64.b64encode(pdf_buffer.read()).decode()
filename = f"Bill_{name.replace(' ', '_')}_{datetime.now().strftime('%d%m%Y_%H%M')}.pdf"


button_html = f"""
<style>
.bill-hero {{
    background: linear-gradient(135deg, #6b7280 0%, #4b5563 50%, #374151 100%);
    padding: 30px;
    border-radius: 25px;
    text-align: center;
    color: black;
    box-shadow: 0 15px 35px rgba(0,0,0,0.2);
    margin: 20px 0;
    font-family: 'Segoe UI', sans-serif;
}}
.btn-download {{
    background: linear-gradient(45deg, #28a745, #20c997);
    color: white !important;
    padding: 18px 50px;
    font-size: 20px;
    border: none;
    border-radius: 50px;
    cursor: pointer;
    margin: 15px;
    font-weight: 700;
    text-decoration: none !important;
    display: inline-block;
    box-shadow: 0 8px 25px rgba(40,167,69,0.4);
    transition: all 0.3s ease;
}}
.btn-download:hover {{
    transform: translateY(-5px);
    box-shadow: 0 12px 35px rgba(40,167,69,0.6) !important;
}}
.btn-print {{
    background: linear-gradient(45deg, #007bff, #0056b3);
    color: white !important;
    padding: 18px 50px;
    font-size: 20px;
    border: none;
    border-radius: 50px;
    cursor: pointer;
    margin: 15px;
    font-weight: 700;
    display: inline-block;
    box-shadow: 0 8px 25px rgba(0,123,255,0.4);
    transition: all 0.3s ease;
}}
.btn-print:hover {{
    transform: translateY(-5px);
    box-shadow: 0 12px 35px rgba(0,123,255,0.6) !important;
}}
.preview-box {{
    background: #f8f9fa;
    border: 2px solid #dee2e6;
    border-radius: 15px;
    padding: 25px;
    margin: 25px 0;
    font-family: 'Courier New', monospace;
    font-size: 14px;
    max-height: 350px;
    overflow-y: auto;
    box-shadow: 0 5px 20px rgba(0,0,0,0.1);
}}
</style>

<div class="bill-hero">
    <h1>🎉 Bill Ready for {name}!</h1>
    <p style="font-size: 18px; margin: 10px 0;">
        Total: <strong>₹{grand_total}</strong> | {order_date}
    </p>
    
    <div class="preview-box">
        <strong>📄 AI Bill Preview:</strong><br><br>
        <pre style="margin: 0; white-space: pre-wrap;">{ai_reply}</pre>
    </div>
    
    <div style="margin: 30px 0;">
        <a href="data:application/pdf;base64,{pdf_base64}" 
           download="{filename}"
           class="btn-download">
            📥 DOWNLOAD YOUR BILL PDF
        </a>
        
        <button class="btn-print" onclick="
            var pdfUrl = 'data:application/pdf;base64,{pdf_base64}';
            var printWin = window.open(pdfUrl, '_blank');
            printWin.onload = function() {{
                printWin.print();
                // printWin.close(); // Uncomment to auto-close
            }};
        ">
            🖨️ PRINT BILL NOW
        </button>
    </div>
    
    <p style="font-size: 14px; opacity: 0.9;">
        💾 Downloads folder | 🖨️ Printer dialog opens automatically
    </p>
</div>

<script>
console.log('Bill buttons loaded! 🎊');
</script>
"""

display(HTML(button_html))

print(f"""
📱 DOWNLOAD: Click green button → PDF saves automatically
🖨️ PRINT: Click blue button 
""")



📱 DOWNLOAD: Click green button → PDF saves automatically
🖨️ PRINT: Click blue button 

